In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_parquet("../../data/processed/processed_taxi_cleaned.parquet")

In [3]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (2451103, 22)

Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'trip_duration_min', 'has_congestion_fee']

First few rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,has_congestion_fee
0,2,2025-01-05 00:00:02,2025-01-05 00:17:47,1.0,8.66,1.0,N,138,43,1,...,0.5,12.85,6.94,1.0,64.24,0.0,1.75,0.0,17.750000,0
1,2,2025-01-05 00:24:51,2025-01-05 00:50:27,1.0,9.73,1.0,N,138,61,1,...,0.5,10.01,0.00,1.0,60.06,0.0,1.75,0.0,25.600000,0
2,2,2025-01-05 00:54:12,2025-01-05 01:17:58,1.0,10.33,1.0,N,138,62,1,...,0.5,10.00,0.00,1.0,60.75,0.0,1.75,0.0,23.766667,0
3,2,2025-01-05 00:10:19,2025-01-05 00:16:03,1.0,1.71,1.0,N,239,24,1,...,0.5,2.86,0.00,1.0,17.16,2.5,0.00,0.0,5.733333,0
4,2,2025-01-05 00:29:32,2025-01-05 00:37:11,1.0,1.80,1.0,N,236,143,1,...,0.5,3.14,0.00,1.0,18.84,2.5,0.00,0.0,7.650000,0


In [4]:
#Load Taxi Zone Lookup Table
#
# The TLC taxi zone lookup maps each LocationID to its borough, zone name,
# and service zone. This is essential for all geographical features.

# Load the taxi zone lookup table
zones = pd.read_csv("../../data/raw/taxi_zone_lookup.csv")

print(f"\nTaxi Zone Lookup shape: {zones.shape}")
print(f"\nColumns: {list(zones.columns)}")
print(f"\nSample rows:")
zones.head(10)


Taxi Zone Lookup shape: (265, 4)

Columns: ['LocationID', 'Borough', 'Zone', 'service_zone']

Sample rows:


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [5]:
# Quick overview of the zone data
print("Boroughs in lookup table:")
print(zones['Borough'].value_counts())
print(f"\nTotal zones: {zones['LocationID'].nunique()}")
print(f"LocationID range: {zones['LocationID'].min()} to {zones['LocationID'].max()}")

Boroughs in lookup table:
Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Unknown           1
Name: count, dtype: int64

Total zones: 265
LocationID range: 1 to 265


In [6]:
# 1. Borough Mapping
#
# Map each PULocationID and DOLocationID to its corresponding borough using
# the taxi zone lookup table.

# %%
# Create borough mapping dictionary from the lookup table
borough_map = zones.set_index('LocationID')['Borough'].to_dict()

# Map boroughs to pickup and dropoff locations
df['pickup_borough'] = df['PULocationID'].map(borough_map)
df['dropoff_borough'] = df['DOLocationID'].map(borough_map)

# Check for any unmapped locations
pu_unmapped = df['pickup_borough'].isna().sum()
do_unmapped = df['dropoff_borough'].isna().sum()
print(f"Unmapped pickup locations: {pu_unmapped:,}")
print(f"Unmapped dropoff locations: {do_unmapped:,}")

# Fill unmapped with 'Unknown' if any exist
if pu_unmapped > 0 or do_unmapped > 0:
    df['pickup_borough'] = df['pickup_borough'].fillna('Unknown')
    df['dropoff_borough'] = df['dropoff_borough'].fillna('Unknown')

# Check which LocationIDs are unmapped
unmapped_do_ids = df.loc[df['DOLocationID'].map(borough_map).isna(), 'DOLocationID'].unique()
print(f"Unmapped DOLocationIDs: {unmapped_do_ids}")

# Display borough distribution for pickups
print("\nPickup Borough Distribution:")
print(df['pickup_borough'].value_counts())

print("\nDropoff Borough Distribution:")
print(df['dropoff_borough'].value_counts())

Unmapped pickup locations: 288
Unmapped dropoff locations: 8,651
Unmapped DOLocationIDs: [265]

Pickup Borough Distribution:
pickup_borough
Manhattan        2234307
Queens            203376
Unknown             6697
Brooklyn            6009
Bronx                633
Staten Island         62
EWR                   19
Name: count, dtype: int64

Dropoff Borough Distribution:
dropoff_borough
Manhattan        2245799
Queens             95185
Brooklyn           78249
Unknown            17515
Bronx               9033
EWR                 4841
Staten Island        481
Name: count, dtype: int64


In [7]:
## 2. Cross-Borough Trip Flags
#
# Identify whether a trip stays within the same borough or crosses borough
# boundaries. Inter-borough trips tend to be longer and more expensive.

# %%
# Same borough flag
df['is_same_borough'] = (df['pickup_borough'] == df['dropoff_borough']).astype(int)

# Create a route borough pair feature (e.g., "Manhattan -> Brooklyn")
df['borough_route'] = df['pickup_borough'] + " -> " + df['dropoff_borough']

# Display summary
print("Cross-Borough Trip Flags Created:")
print(f"  - is_same_borough: {df['is_same_borough'].sum():,} trips ({df['is_same_borough'].mean()*100:.1f}%)")

print("\nTop 10 Borough Routes:")
print(df['borough_route'].value_counts().head(10))

Cross-Borough Trip Flags Created:
  - is_same_borough: 2,165,565 trips (88.4%)

Top 10 Borough Routes:
borough_route
Manhattan -> Manhattan    2118385
Queens -> Manhattan        122082
Manhattan -> Queens         54157
Manhattan -> Brooklyn       44337
Queens -> Queens            40260
Queens -> Brooklyn          30426
Manhattan -> Unknown         7882
Queens -> Unknown            6336
Manhattan -> Bronx           4769
Manhattan -> EWR             4575
Name: count, dtype: int64


In [8]:
## 3. Airport Flags
#
# Create flags for trips originating from or destined to major NYC airports.
# Airport trips often have flat-rate fares and distinct patterns.
#
# **Airport Zone IDs (TLC):**
# - JFK Airport: LocationID 132
# - LaGuardia Airport: LocationID 138
# - Newark/EWR: LocationID 1

# %%
# Define airport location IDs
AIRPORT_ZONES = {
    'JFK': 132,
    'LaGuardia': 138,
    'EWR': 1       # Newark Airport
}
ALL_AIRPORT_IDS = list(AIRPORT_ZONES.values())

# --- Airport pickup flags ---
df['is_airport_pickup'] = df['PULocationID'].isin(ALL_AIRPORT_IDS).astype(int)
df['is_jfk_pickup'] = (df['PULocationID'] == AIRPORT_ZONES['JFK']).astype(int)
df['is_lga_pickup'] = (df['PULocationID'] == AIRPORT_ZONES['LaGuardia']).astype(int)
df['is_ewr_pickup'] = (df['PULocationID'] == AIRPORT_ZONES['EWR']).astype(int)

# --- Airport dropoff flags ---
df['is_airport_dropoff'] = df['DOLocationID'].isin(ALL_AIRPORT_IDS).astype(int)
df['is_jfk_dropoff'] = (df['DOLocationID'] == AIRPORT_ZONES['JFK']).astype(int)
df['is_lga_dropoff'] = (df['DOLocationID'] == AIRPORT_ZONES['LaGuardia']).astype(int)
df['is_ewr_dropoff'] = (df['DOLocationID'] == AIRPORT_ZONES['EWR']).astype(int)

# --- Combined airport trip flag (either pickup or dropoff at an airport) ---
df['is_airport_trip'] = (df['is_airport_pickup'] | df['is_airport_dropoff']).astype(int)

# Display summary
print("Airport Flags Created:")
print(f"  - Airport pickups:  {df['is_airport_pickup'].sum():,} trips ({df['is_airport_pickup'].mean()*100:.1f}%)")
print(f"      JFK:            {df['is_jfk_pickup'].sum():,}")
print(f"      LaGuardia:      {df['is_lga_pickup'].sum():,}")
print(f"      EWR/Newark:     {df['is_ewr_pickup'].sum():,}")
print(f"  - Airport dropoffs: {df['is_airport_dropoff'].sum():,} trips ({df['is_airport_dropoff'].mean()*100:.1f}%)")
print(f"      JFK:            {df['is_jfk_dropoff'].sum():,}")
print(f"      LaGuardia:      {df['is_lga_dropoff'].sum():,}")
print(f"      EWR/Newark:     {df['is_ewr_dropoff'].sum():,}")
print(f"  - Total airport trips: {df['is_airport_trip'].sum():,} ({df['is_airport_trip'].mean()*100:.1f}%)")


Airport Flags Created:
  - Airport pickups:  187,358 trips (7.6%)
      JFK:            112,830
      LaGuardia:      74,509
      EWR/Newark:     19
  - Airport dropoffs: 47,866 trips (2.0%)
      JFK:            18,844
      LaGuardia:      24,181
      EWR/Newark:     4,841
  - Total airport trips: 230,122 (9.4%)


In [9]:
## 4. Zone Popularity / Frequency Encoding
#
# Encode each pickup and dropoff zone by how frequently it appears in the
# dataset. High-frequency zones (e.g., Midtown, Times Square) behave
# differently from low-frequency zones. We also encode route-pair frequency.

# %%
# --- Pickup zone frequency ---
pu_freq = df['PULocationID'].value_counts(normalize=True)
df['pickup_zone_freq'] = df['PULocationID'].map(pu_freq)

# --- Dropoff zone frequency ---
do_freq = df['DOLocationID'].value_counts(normalize=True)
df['dropoff_zone_freq'] = df['DOLocationID'].map(do_freq)

# Display top pickup zones by frequency
print("Top 10 Most Popular Pickup Zones:")
top_pu = (
    df[['PULocationID', 'pickup_zone_freq']]
    .drop_duplicates()
    .sort_values('pickup_zone_freq', ascending=False)
    .head(10)
)
# Merge zone names for readability
top_pu = top_pu.merge(zones[['LocationID', 'Zone', 'Borough']], 
                       left_on='PULocationID', right_on='LocationID', how='left')
print(top_pu[['PULocationID', 'Zone', 'Borough', 'pickup_zone_freq']].to_string(index=False))

print("\nTop 10 Most Popular Dropoff Zones:")
top_do = (
    df[['DOLocationID', 'dropoff_zone_freq']]
    .drop_duplicates()
    .sort_values('dropoff_zone_freq', ascending=False)
    .head(10)
)
top_do = top_do.merge(zones[['LocationID', 'Zone', 'Borough']], 
                       left_on='DOLocationID', right_on='LocationID', how='left')
print(top_do[['DOLocationID', 'Zone', 'Borough', 'dropoff_zone_freq']].to_string(index=False))


Top 10 Most Popular Pickup Zones:
 PULocationID                         Zone   Borough  pickup_zone_freq
          237        Upper East Side South Manhattan          0.054746
          161               Midtown Center Manhattan          0.053582
          236        Upper East Side North Manhattan          0.051029
          132                  JFK Airport    Queens          0.046032
          162                 Midtown East Manhattan          0.038673
          186 Penn Station/Madison Sq West Manhattan          0.038605
          230    Times Sq/Theatre District Manhattan          0.038121
          142          Lincoln Square East Manhattan          0.034917
          163                Midtown North Manhattan          0.030583
          138            LaGuardia Airport    Queens          0.030398

Top 10 Most Popular Dropoff Zones:
 DOLocationID                      Zone   Borough  dropoff_zone_freq
          236     Upper East Side North Manhattan           0.053333
          2

In [10]:
#  --- Route-pair frequency encoding ---
# Create a route key from pickup -> dropoff zone pair
df['route_id'] = df['PULocationID'].astype(str) + "_" + df['DOLocationID'].astype(str)

route_freq = df['route_id'].value_counts(normalize=True)
df['route_freq'] = df['route_id'].map(route_freq)

print("Route-Pair Frequency Encoding:")
print(f"  Unique routes: {df['route_id'].nunique():,}")
print(f"  Mean route frequency: {df['route_freq'].mean():.6f}")
print(f"  Median route frequency: {df['route_freq'].median():.6f}")
print(f"  Max route frequency: {df['route_freq'].max():.6f}")

print("\nTop 10 Most Common Routes:")
top_routes = (
    df[['route_id', 'route_freq']]
    .drop_duplicates()
    .sort_values('route_freq', ascending=False)
    .head(10)
)
# Split route_id back for readability
top_routes[['PU_ID', 'DO_ID']] = top_routes['route_id'].str.split('_', expand=True).astype(int)
top_routes = top_routes.merge(zones[['LocationID', 'Zone']], left_on='PU_ID', right_on='LocationID', how='left').rename(columns={'Zone': 'PU_Zone'})
top_routes = top_routes.merge(zones[['LocationID', 'Zone']], left_on='DO_ID', right_on='LocationID', how='left').rename(columns={'Zone': 'DO_Zone'})
print(top_routes[['PU_Zone', 'DO_Zone', 'route_freq']].to_string(index=False))


Route-Pair Frequency Encoding:
  Unique routes: 14,839
  Mean route frequency: 0.001040
  Median route frequency: 0.000654
  Max route frequency: 0.008782

Top 10 Most Common Routes:
              PU_Zone               DO_Zone  route_freq
Upper East Side South Upper East Side North    0.008782
Upper East Side North Upper East Side South    0.007616
Upper East Side North Upper East Side North    0.006057
Upper East Side South Upper East Side South    0.005767
       Midtown Center Upper East Side South    0.004019
Upper East Side South        Midtown Center    0.003483
       Midtown Center Upper East Side North    0.003428
Upper West Side South Upper West Side North    0.003211
  Lincoln Square East Upper West Side South    0.003063
      Lenox Hill West Upper East Side North    0.002958


In [11]:
## Summary of Created Features
#
# Let's list all the geographical features we've created:

# %%
# List all geographical features created
geo_features = [
    'pickup_borough',
    'dropoff_borough',
    'is_same_borough',
    'borough_route',
    'is_airport_pickup',
    'is_jfk_pickup',
    'is_lga_pickup',
    'is_ewr_pickup',
    'is_airport_dropoff',
    'is_jfk_dropoff',
    'is_lga_dropoff',
    'is_ewr_dropoff',
    'is_airport_trip',
    'pickup_zone_freq',
    'dropoff_zone_freq',
    'route_id',
    'route_freq',
]

print(f"Total geographical features created: {len(geo_features)}")
print("\nFeature List:")
for i, feat in enumerate(geo_features, 1):
    print(f"  {i:2d}. {feat}")

# Display data types
print("\n\nFeature Data Types:")
print(df[geo_features].dtypes)

Total geographical features created: 17

Feature List:
   1. pickup_borough
   2. dropoff_borough
   3. is_same_borough
   4. borough_route
   5. is_airport_pickup
   6. is_jfk_pickup
   7. is_lga_pickup
   8. is_ewr_pickup
   9. is_airport_dropoff
  10. is_jfk_dropoff
  11. is_lga_dropoff
  12. is_ewr_dropoff
  13. is_airport_trip
  14. pickup_zone_freq
  15. dropoff_zone_freq
  16. route_id
  17. route_freq


Feature Data Types:
pickup_borough            str
dropoff_borough           str
is_same_borough         int64
borough_route             str
is_airport_pickup       int64
is_jfk_pickup           int64
is_lga_pickup           int64
is_ewr_pickup           int64
is_airport_dropoff      int64
is_jfk_dropoff          int64
is_lga_dropoff          int64
is_ewr_dropoff          int64
is_airport_trip         int64
pickup_zone_freq      float64
dropoff_zone_freq     float64
route_id                  str
route_freq            float64
dtype: object


In [12]:
#  Save the enhanced dataset
output_path = "../../data/processed/processed_taxi_with_geo_features.parquet"
df.to_parquet(output_path, index=False)

print(f"Dataset saved to: {output_path}")
print(f"Final dataset shape: {df.shape}")
print(f"Total features: {len(df.columns)}")

Dataset saved to: ../../data/processed/processed_taxi_with_geo_features.parquet
Final dataset shape: (2451103, 39)
Total features: 39
